# EDA of YouTube Dataset

In [1]:
import pandas as pd
import glob

# Find all CSV files or list them explicitly
csv_files = glob.glob("*.csv")  # Assumes they are in the same folder
dataframes = {}

for file in csv_files:
    # Read data, treating common missing value strings as NaN
    dataframes[file] = pd.read_csv(file, na_values=["NA", "N/A", "missing", ""])
    print(f"Loaded {file} - Shape: {dataframes[file].shape}")


Loaded daily_metrics.csv - Shape: (2526, 6)
Loaded geography.csv - Shape: (20, 3)
Loaded traffic_sources.csv - Shape: (15, 3)
Loaded videos.csv - Shape: (107, 9)


In [2]:
def clean_dataframe(df):
    # 1. Clean column names
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    
    # 2. Drop absolute duplicates
    df = df.drop_duplicates()
    
    # 3. Handle specific missing data (example)
    # df['status'] = df['status'].fillna('unknown')
    
    return df

# Apply to all loaded files
cleaned_dfs = {name: clean_dataframe(df) for name, df in dataframes.items()}

In [3]:
for filename, df in cleaned_dfs.items():
    print(f"--- Columns in {filename} ---")
    print(df.columns.tolist())
    print()

--- Columns in daily_metrics.csv ---
['day', 'views', 'estimatedminuteswatched', 'averageviewduration', 'subscribersgained', 'subscriberslost']

--- Columns in geography.csv ---
['country', 'views', 'estimatedminuteswatched']

--- Columns in traffic_sources.csv ---
['insighttrafficsourcetype', 'views', 'estimatedminuteswatched']

--- Columns in videos.csv ---
['video_id', 'title', 'published_at', 'tags', 'category_id', 'duration_seconds', 'views', 'likes', 'comments']



In [4]:
# Print the columns so we can see the exact spelling
print("Your actual columns in daily_metrics.csv are:")
print(cleaned_dfs['daily_metrics.csv'].columns.tolist())

Your actual columns in daily_metrics.csv are:
['day', 'views', 'estimatedminuteswatched', 'averageviewduration', 'subscribersgained', 'subscriberslost']


In [5]:
# 1. Grab the dataframe
metrics_df = cleaned_dfs['daily_metrics.csv']

# 2. Run validations using the exact column names
assert (metrics_df['views'] >= 0).all(), "Negative view counts detected!"
assert (metrics_df['estimatedminuteswatched'] >= 0).all(), "Negative watch time detected!"
assert (metrics_df['averageviewduration'] >= 0).all(), "Negative view duration detected!"
assert (metrics_df['subscribersgained'] >= 0).all(), "Negative subscribers gained detected!"
assert (metrics_df['subscriberslost'] >= 0).all(), "Negative subscribers lost detected!"

print("🎉 All validations for daily_metrics.csv passed successfully!")

🎉 All validations for daily_metrics.csv passed successfully!


In [6]:
for file in ['videos.csv', 'geography.csv', 'traffic_sources.csv']:
    print(f"Columns in {file}:")
    print(cleaned_dfs[file].columns.tolist())
    print("-" * 30)

Columns in videos.csv:
['video_id', 'title', 'published_at', 'tags', 'category_id', 'duration_seconds', 'views', 'likes', 'comments']
------------------------------
Columns in geography.csv:
['country', 'views', 'estimatedminuteswatched']
------------------------------
Columns in traffic_sources.csv:
['insighttrafficsourcetype', 'views', 'estimatedminuteswatched']
------------------------------


In [7]:
# Validate videos
videos_df = cleaned_dfs['videos.csv']
assert videos_df['video_id'].is_unique, "Duplicate video_ids found!"
assert (videos_df[['views', 'likes', 'comments']] >= 0).all().all(), "Negative metrics in videos!"

# Validate geography & traffic
assert (cleaned_dfs['geography.csv']['views'] >= 0).all(), "Negative views in geography!"
assert (cleaned_dfs['traffic_sources.csv']['views'] >= 0).all(), "Negative views in traffic sources!"

print("🎉 All files successfully validated!")

🎉 All files successfully validated!


In [8]:
import pandas as pd
import glob

# 1. Load the files
csv_files = glob.glob("*.csv")  
dataframes = {}

for file in csv_files:
    dataframes[file] = pd.read_csv(file, na_values=["NA", "N/A", "missing", ""])

# 2. Clean the files (this creates the missing 'cleaned_dfs' variable)
cleaned_dfs = {}
for name, df in dataframes.items():
    # Standardize column names
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    df = df.drop_duplicates()
    cleaned_dfs[name] = df

print("Data successfully reloaded and cleaned! 'cleaned_dfs' is now ready.")

Data successfully reloaded and cleaned! 'cleaned_dfs' is now ready.


## 1. Find Your Top-Performing Videos

In [9]:
videos_df = cleaned_dfs['videos.csv']

# Calculate engagement rate (handling division by zero just in case)
videos_df['engagement_rate'] = (videos_df['likes'] / videos_df['views']).fillna(0) * 100

# Get top 5 videos by views
top_videos = videos_df.sort_values(by='views', ascending=False).head(5)
print("--- TOP 5 VIDEOS BY VIEWS ---")
print(top_videos[['title', 'views', 'likes', 'engagement_rate']])

--- TOP 5 VIDEOS BY VIEWS ---
                                                title  views  likes  \
28             #trampoline #ritzhappyworld #cityofjoy   2459     31   
70                  Catch the Specs 😶😶#trendingshorts   1253     21   
86  KOLKATA TO BOKARO| KOLKATA TO BOKARO BY CAR | ...    754     29   
41  मैं तोह उड़ने लगी 🐦😂 #ritzhappyworld #manali #h...    508      8   
23   #WetNWildAdventures #NiccoParkFun#ritzhappyworld    456     12   

    engagement_rate  
28         1.260675  
70         1.675978  
86         3.846154  
41         1.574803  
23         2.631579  


## 3. Top Traffic Sources & Geographies

In [10]:
geo_df = cleaned_dfs['geography.csv']
traffic_df = cleaned_dfs['traffic_sources.csv']

print("--- TOP 3 COUNTRIES BY VIEWS ---")
# Fixed .header(3) to .head(3)
print(geo_df.sort_values(by='views', ascending=False).head(3)[['country', 'views']])

print("\n--- TOP 3 TRAFFIC SOURCES ---")
# Using the original column name found in your dataset
print(traffic_df.sort_values(by='views', ascending=False).head(3)[['insighttrafficsourcetype', 'views']])

--- TOP 3 COUNTRIES BY VIEWS ---
  country  views
0      IN   8681
1      BD    172
2      PK    164

--- TOP 3 TRAFFIC SOURCES ---
  insighttrafficsourcetype  views
0                   SHORTS   8413
1                YT_SEARCH   3209
2                  EXT_URL    944


In [ ]:
import pandas as pd
import glob
import matplotlib
# FORCE OFFLINE BACKEND (Prevents graphics-card related kernel crashes)
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Reload data into memory safely
csv_files = glob.glob("*.csv")  
dataframes = {}
for file in csv_files:
    dataframes[file] = pd.read_csv(file, na_values=["NA", "N/A", "missing", ""])

cleaned_dfs = {}
for name, df in dataframes.items():
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    df = df.drop_duplicates()
    cleaned_dfs[name] = df

# 2. Extract specific dataframes
metrics_df = cleaned_dfs['daily_metrics.csv'].copy()
videos_df = cleaned_dfs['videos.csv'].copy()
traffic_df = cleaned_dfs['traffic_sources.csv'].copy()
geo_df = cleaned_dfs['geography.csv'].copy()

# 3. Format dates safely
metrics_df['day'] = pd.to_datetime(metrics_df['day'], errors='coerce')
metrics_df = metrics_df.dropna(subset=['day'])
metrics_sorted = metrics_df.sort_values('day')

# 4. Set professional styling
sns.set_theme(style="whitegrid")

print("Processing charts safely... (No windows will pop up)")

# ==========================================
# CHART 1: Daily Views Trend Over Time
# ==========================================
plt.figure(figsize=(14, 4))
sns.lineplot(data=metrics_sorted, x='day', y='views', color='#1676F3', linewidth=2.5)
plt.title('Daily Channel Views Trend Over Time', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Total Daily Views', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('1_daily_views_trend.png', dpi=300) # SAVES SILENTLY TO FILE
plt.close()

# ==========================================
# CHART 2: Top 10 Most Viewed Videos
# ==========================================
plt.figure(figsize=(12, 6))
top_10_videos = videos_df.sort_values(by='views', ascending=False).head(10)
sns.barplot(data=top_10_videos, x='views', y='title', palette='viridis', hue='title', legend=False)
plt.title('Top 10 Most Viewed Videos on the Channel', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Total Cumulative Views', fontsize=12)
plt.ylabel('')
plt.tight_layout()
plt.savefig('2_top_10_videos.png', dpi=300) # SAVES SILENTLY TO FILE
plt.close()

# ==========================================
# CHART 3: Traffic Source Breakdown
# ==========================================
plt.figure(figsize=(9, 5))
traffic_sorted = traffic_df.sort_values(by='views', ascending=False)
sns.barplot(data=traffic_sorted, x='insighttrafficsourcetype', y='views', palette='Blues_r', hue='insighttrafficsourcetype', legend=False)
plt.title('Where Your Views Come From (Traffic Sources)', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Traffic Channel Type', fontsize=12)
plt.ylabel('Views', fontsize=12)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('3_traffic_sources.png', dpi=300) # SAVES SILENTLY TO FILE
plt.close()

# ==========================================
# CHART 4: Geographic Distribution of Views
# ==========================================
plt.figure(figsize=(9, 5))
geo_sorted = geo_df.sort_values(by='views', ascending=False)
sns.barplot(data=geo_sorted, x='country', y='views', palette='Reds_r', hue='country', legend=False)
plt.title('Audience Distribution by Country', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Country Code', fontsize=12)
plt.ylabel('Views', fontsize=12)
plt.tight_layout()
plt.savefig('4_geographic_distribution.png', dpi=300) # SAVES SILENTLY TO FILE
plt.close()

print("🎉 Success! Check your project folder. You will find 4 beautiful image files waiting for you:")
print("   - 1_daily_views_trend.png")
print("   - 2_top_10_videos.png")
print("   - 3_traffic_sources.png")
print("   - 4_geographic_distribution.png")